# Chapter 27: Data Wrangling
**Author:** Meghan R. Hutch

**focus on date-time formatting**

**median impuation - keeping it simple**

**add datasets as footnotes like Amandas case-study**

### What does it mean to wrangle our data?

Simply, data wrangling is the act of preparing data for analysis. Many of the datasets you work with during class may already be fairly or completely clean. Meaning, the data was previously prepared to make it easy for you to download and begin analyzing right away. Most real-word data is messy due to the way the data was collected. For instance, there might be missing values, measurements might be in different units, text-labeles might have typos or varying use of uppercase or lowercase letters. Thus, it is critical to check for, and resolve, any inconsistencies in our data prior to analysis. 

This task if often not trivial and requires careful investigation and consultation with domain experts - those who can help clarify how the data was collected and what variables mean. This process also helps ensures that downstream analyses will not be hindered by data inaccuracies. Thus, we can feel confident about the conclusions we draw as they relate to the question or problem we are trying to solve.

> “Data science, [...] involves multiple “very small decisions” — data cleaning and filtering steps, for instance, which are crucially important, but difficult to document. And journal page limits preclude exposition. But by blending code, data and text in a single document, researchers can show just how their results were generated.” - [Ben Marwick Perkel, JM](https://www.nature.com/articles/d41586-022-00563-z)


## How to begin Wrangling: Get to know your Data!

Whenever you begin working on a new project or with any new dataset, it's essential that you get to know your data. To get started, there are a few questions you might ask yourself:

* Where did the data come from and how it was collected?

* What data types do I have?

* Does Python recognize these correctly? 

### Case Study of Data Wrangling

Through this chapter, we will begin answering these questions by working with a new real-world dataset called MIMIC.The [Medical Information Mart for Intensive Care (MIMIC)-IV](https://physionet.org/content/mimic-iv-demo/) is a large dataset curated to help support studies on intensive care unit (ICU) patients.

Medical data is a great case study for the importance of data wrangling. Medical records often contain heterogenous types of data and concepts and measurements can be recorded in unstandardized ways. 

#### Where did the data come from and how was it collected?

The MIMIC-IV dataset (since April 2024) contains structured EHR data from >360,000 patients who were admitted to Beth Israel Deaconess Medical Center in Boston, MA between 2008-2022. The dataset contains information that was entered into a patient's medical record when they were admitted to the hospital.

For this case-study, we will rely on the demo-dataset which has been made publically available and contains the data from random subset of 100 hospitalized patients.*

The MIMIC-IV dataset contains many seperate data tables (structured as csv files). To begin, we will import one of the csv files concerning the demographics of the patients included in study.
  
***Note**: For the purposes of this chapter, we will use a reduced subset of the demo-data.*

In [11]:
import pandas as pd

patients = pd.read_csv('data/mimic_patients.csv')

patients

,subject_id,gender,anchor_age
0,10014729,F,21
1,10003400,F,72
2,10002428,F,80
3,10032725,F,38
4,10027445,F,48
...,...,...,...
95,10004733,M,51
96,10021118,M,62
97,10018501,M,83
98,10007058,M,48


#### Determine Data Types

Now that we've loaded in our data, we can start to explore what type of data we have. `head()` can be a good first choice for examining the first few rows of the data.

We see that we have three columns: `subject_id` (unique patient identifier), `gender`, and `anchor_age` (age of the patient when admitted). 

### Side-Note: Data Dictionaries and Codebooks

The MIMIC-IV dataset curations have curated excellent [documentation](https://mimic.mit.edu/docs/) regarding the information about all of the tables available in the MIMIC dataset and the variables each table contains. When working with datasets, it's good practice to determine whether there is a data dictionary or codebook which might tell us what variables represent and what data types they are. For example, we can find information about the patients table in the [documentation's patients table page](https://mimic.mit.edu/docs/iv/modules/hosp/patients/).

The documentation tells us the following:
* `anchor_age` is the patient’s age in the `anchor_year` (note: this column is not included in our demo-set). If a patient’s `anchor_age` is over 89 in the anchor_year then their `anchor_age` is set to 91, regardless of how old they actually were.
* Example: a patient has an `anchor_year` of 2153, `anchor_year_group` of 2008 - 2010, and an `anchor_age` of 60.
    * The year 2153 for the patient corresponds to 2008, 2009, or 2010.
    * The patient was 60 in the shifted year of 2153, i.e. they were 60 in 2008, 2009, or 2010.
    * A patient admission in 2154 will occur in 2009-2011, an admission in 2155 will occur in 2010-2012, and so on.

These details are helpful because this information on `anchor_age` is clearly not intuitive from just the column name alone. 

In [ ]:
demographics.head()

Since `head()` only showed us the first 5 rows of data, let's run some code to evaluate how many rows we have.

In [ ]:
len(demographics)

We should also check that `subject_id` is in-fact unique - if it is, the number of unique values for `subject_id` should equal the number of rows in our dataframe.

In [ ]:
demographics['subject_id'].nunique()

In [ ]:
# we could also automate this check
demographics['subject_id'].nunique() == len(demographics)

Next, we might want to check how Python interepreted these data types. 

In [ ]:
demographics.dtypes

Should `subject_id` really be considered an integer? We are informed from the dataset curators that this variable is a unique subject (or patient) identifier. 

Ask yourself: would adding two `subject_id` values make sense?

If this is a unique identifier, we can almost think of this variable representing a person's name. 

Let's convert it to a string to make sure we don't accidently try to include it as a variable in subsequent computations.

In [ ]:
demographics['subject_id'] = demographics['subject_id'].astype('str')

demographics.dtypes

From this step, we also see that `gender` is of type object and `anchor_age` is of type integer. These seem reasonable!

#### Examine the Values of Variables

To understand our data even more, we can inspect the actual values of our variables. For example, what categorical labels is the `gender` variable composed of?

In [ ]:
demographics['gender'].value_counts()

For numerical variables, we can access the distribution of values by calculating summary statictics.

In [ ]:
demographics['anchor_age'].describe()

Patients in our data have a mean range of 61.8. The youngest and oldest patients are 21 and 91 years old, respectively. 

Better yet, for a quick glance of the distribution, we can also use a histogram:

In [ ]:
demographics['anchor_age'].hist()

Checking the distribution of numerical values is important as it can help us quickly see if there is an unexpected observation in our data. For instance, perhaps we were told our dataset should only include adults, but we saw some Ages that were < 10 years old. Or maybe someone is said to be 150 years old! These would necessitate further exploration to determine whether there are any data entry issues. Perhaps a busy doctor mistyped the age or maybe age was confused with a different variable.

#### Incorporating more data!

As previously mentioned, MIMIC contains multiple csv files worth of data. Next, let's look at a second file: vitals.csv which contains measurements of the patient.

In [ ]:
vitals = pd.read_csv('data/mimic_vitals.csv')

##### Examine the Data

Let's repeat the previously described steps to help us get to know our new `vitals` dataframe

In [ ]:
vitals.head()

Once again, we see that the `subject_id` column is present, in addition to three new columns named: `chartdate`, `result_name`, and `result_value`.

Let's see if the number of unique `subject_id`s match the number of rows:

In [ ]:
vitals['subject_id'].nunique() == len(vitals)

In [ ]:
len(vitals)

In this case, it appears that there are many more observations of vitals (note: this was also apparent just from looking at the first 5 rows: the same `subject_id` appears in each row!). 

To get a better idea of what is going on here, we can examine the specific types of categories in the `result_name` column. When we do, we leaarn that this variables captures measurements including weight, height, BMI, and blood pressure. It seems reasonable that patients may have each had these taken several times during any single or maybe multiple hospital visits.

In [ ]:
vitals['result_name'].value_counts()

As we did before, let's also check the data types:

In [ ]:
vitals.dtypes

First, we can convert `subject_id` to a string again

In [ ]:
vitals['subject_id'] = vitals['subject_id'].astype('str')

Next, we see that `chartdate` is a object. As previously mentioned in the text [**add link**], we can convert dates to a special datetime data type in Python. This specification will allow us to perform operations between dates (e.g., finding the number of days between two blood pressure measurements).

In [ ]:
vitals['chartdate'] = pd.to_datetime(vitals['chartdate'])
vitals

#### Important Note about Dates

`chartdate` is the date the vital sign was recorded. You'll notice that these years are in the future. That's due to privacy protections that were put in place when releasing the data; dates were shifted up randomly for each patient (though each patient's dates were shifted a consistent amount. Thus, if two blood presure measurements were taken on two consequentive dates, the shifted dates might look like: 2067-01-01 and 2067-01-02). Knowing whether or not a dataset's dates are accurate or have been shifted is crucial if you're trying to draw conclusions that are based on the actual date. For example, let's say a patient admitted with an unknown but severe infection in 1990, but had their dates shifted to 2020 - it would be completely erroneous to conclude they may have had COVID-19! 



Interestingly, result name is not being recognized as an integer/numeric. Perhaps there are non-numeric values in this column? We can check this using the following expression to see which observations have non-numeric characters: 

In [ ]:
# return the rows where there are non-numeric characters in `result_value`
vitals[vitals['result_value'].astype(str).str.contains(r'[^0-9]')]

We see that some observations is composed of numbers appearing like floats and strings where `result_name` == 'Blood Pressure` is expressed with a '/'. 

If we were interested in investigating blood pressure of these patients, we'd need to be mindful that Python will not allow us to simply average blood pressure without doing further data manipulation. 

#### Wrangling numbers in Strings

In [ ]:
# example of preparing blood pressurte

#### Filtering Observations of Interest

Perhaps we are only interested in height and weight. We can create a new copy of the vitals dataframe and filter it to keep those observations of interest.

In [ ]:
height = vitals.copy()
height = height[height['result_name'] == 'Height (Inches)']

weight = vitals.copy()
weight = vitals_copy[vitals_copy['result_name'] == 'Weight (Lbs)']

In [ ]:
print('# Height Observations:', len(height))
print('# Weight Observations:', len(weight))

#### Evaluate distribution of height

It looks like `result_value` is a continuous variable. Let's make sure that Python also has recognized it as such.

In [ ]:
height['result_value'] = pd.to_numeric(height['result_value'])

In [ ]:
# check variable data types
height.dtypes

#### Identify Outliers using Summary Statistics

Next, let's calculate summary statistics to get a better sense of our variable

In [ ]:
# how many unique patients have a height recording
# 61 patients - not all of our 100 patients had a record for height
height.nunique()

In [ ]:
# how many observations - 378 observations, thus some patients must have height recorded more than once
len(height)

**Visualize distribution of values**

In [ ]:
height['result_value'].hist()

In [ ]:
# use describe() to compute summary statistics of the distribution
height['result_value'].describe()

**Potential red flag:** Is there really a patient who might be 5 inches tall? This is a clear case I'd like to investigate more. Perhaps this is a data entry issue?

First, we can sort the height values from lowest to highest in order to retrieve the `subject_id` of the patient with a height of 5 inches

I can also see if there are any other patients with unusually low heights

In [ ]:
height.sort_values(['result_value'])

Let's see if this patient has other recorded height measurements

In [ ]:
height[height['subject_id']=='10012853']

Interesting - it appears this patient had an initial obsevation of 5 inches, but then three years later was always measured at 64 inches (or 5'4 in feet/inches). It seems even further unlikely that a person would have that dramatic of a growth in such a short timespan. But, let's investigate the variation in change for other patients who have more than one height measurement.

For each unique patient, we will calculate the standard deviation of the patient's height measurements

*Note*: `ddof = 0` indicates that std for patients with one measurement will be displayed as 0 rather than NaN

In [ ]:
import numpy as np

height['std'] = height.groupby('subject_id')['result_value'].transform(np.std, ddof=0)

In [ ]:
height.sort_values('std', ascending = False)[['subject_id', 'std']].drop_duplicates().head(10)

We can clearly see the one 'concerning' patient has a much higher standard deviation of their height measurements compared to other patients.

Let's take a look at the two patients with the second and third highest standard deviation for comparison

In [ ]:
height[height['subject_id']=='10005909']

In [ ]:
height[height['subject_id']=='10019917']

It clearly seems that there the 5 inch observation is an outlier. Perhaps whoever recorded the data mistook the variable for feet instead of inches.

The best way to handle these types of inconsistencies will largely depend on the exact variable, question of interest, and domain knowledge. In this case, it may be safe to take the median (64 years) for the concerning observation since this was a consistent height recorded over the following years. However, how there are a couple of ways to best handle apparent outliers:

* Seek consultation from mentors and domain experts! 

* Consider the age of the patient - height may be expected to change more dramatically over time.

* Depending on the patient's age, it may be fair to take the next closest measurement in time.

#### Handling multiple observations

In this case, let's see what happens if we group by `subject_id` and take the median height in cases when a patient has multiple measurements

In [ ]:
# create a new variable with the median height value for each patient
height['result_value_median'] = height.groupby('subject_id')['result_value'].transform('median')

This next step ensures that we will now only have one unique row per patient

In [ ]:
# assuming we don't need to link by date
height_clean = height[['subject_id', 'result_name', 'result_value_median']].drop_duplicates()

In [ ]:
# should have one observation per person
len(height_clean)

Let's review the distribution again - now it looks a bit more normally distributed

In [ ]:
height_clean['result_value_median'].hist()

#### Unit Conversion Issues

In [ ]:
weight.head()

**Review variable types**

It looks like `result_value` is a continuous variable. Let's make sure that Python also has recognized it as such.

In [ ]:
# check variable data types
weight.dtypes

In [ ]:
weight['result_value'] = pd.to_numeric(weight['result_value'])

In [ ]:
# check variable data types
weight.dtypes

In [ ]:
weight['result_value'].describe()

**Distribution of weight values**

In [ ]:
weight['result_value'].hist()

In [ ]:
weight.sort_values(['result_value'])

**Investigate the variance of in measurements between patients with multiple observations**

In [ ]:
weight['std'] = weight.groupby('subject_id')['result_value'].transform(np.std, ddof=0)# ddof = 0 indicates that std for patients with one measurement will be displayed as 0 rather than NaN

In [ ]:
weight.sort_values('std', ascending = False)[['subject_id', 'std']].drop_duplicates().head(5)

In [ ]:
weight[weight['subject_id']=='10019385'] # did this patient lose > 110 pounds in < 1 month?

**Potential Unit Issue? 97 kg ~ 214 pounds**

In [ ]:
weight[weight['subject_id']=='10021487'].sort_values('chartdate')

In the first case, my intuition is that the discrepancy in weight measurements is due to a unit conversions issue: 97 kg ~ 214 pounds

In the second case, the patient has many weight measurements over the a 1.5 year period. They are not varrying too widely. It looks like there is an additional weight loss, followed by an increase in weight (U-shaped curve).

#### Merging Data from seperate tables

In [ ]:
# demo_vitals = pd.merge(demographics,
#                        height,
#                        on = 'subject_id',
#                        how = 'left')

# demo_vitals = pd.merge(demo_vitals,
#                        weight,
#                        on = 'subject_id',
#                        how = 'left')

#### Evaluating Unit Conversions

#### Missing Data

This topic could be it's own class. 

[Missing/missing at random]

Rule of thumb: Talk to your domain experts/collaborators!

Some common ways to handle missing data:

* Remove those patients
* Impute mean or median
* Forward/backward filling
* MICE